# Customer-Support Assistant - Ticket Urgency

**What this notebook does:** this is where we experiment with the assistant's ticket-urgency detector. As of Lesson 15 it is a real **learned classifier** — logistic regression — that reads each ticket's count of urgency-signal words and outputs a **probability** that the ticket is urgent, then a yes/no decision.

The clean, runnable version of this same logic lives one folder up as `../ticket_urgency.py`, wired into `../main.py`. This notebook is for reading and experimenting one cell at a time.

This replaces the original hand-written word-score rule from Lesson 01. Instead of a human hard-coding "score above zero means urgent," the model now **learns** from labelled tickets how strongly the signal-word count implies urgency, and where the decision boundary belongs.

## The training examples

These stand in for real historical tickets the store might have on file. Each one is labeled by a human as "urgent" or "not urgent." This labeled set is what the assistant learns from.

In [2]:
training_tickets = [
    ("My order still has not arrived and I need it right now", "urgent"),
    ("This is extremely time sensitive please help right now", "urgent"),
    ("The app crashed and I cannot log in please fix this immediately", "urgent"),
    ("My package was supposed to arrive yesterday and now it says lost", "urgent"),
    ("Just wondering when my package will ship no rush", "not urgent"),
    ("Can you tell me more about your return policy sometime", "not urgent"),
    ("I have a general question about sizing whenever you get a chance", "not urgent"),
    ("Do you offer gift wrapping for orders", "not urgent"),
]

In [3]:
import math

# Words that hint a ticket might be urgent. Counting these is our one feature.
SIGNAL_WORDS = {
    "now", "immediately", "right", "today", "asap", "urgent",
    "cannot", "help", "fix", "lost", "emergency", "crashed",
}

def tokenize(text):
    lowered = text.lower().replace(",", "").replace(".", "")   # drop case and punctuation
    return lowered.split()                                     # split into words

def count_signal_words(text):
    words = tokenize(text)                                     # the ticket's words
    count = 0                                                  # running tally
    for word in words:
        if word in SIGNAL_WORDS:                              # a signal word?
            count += 1
    return count

def sigmoid(z):
    # Squash any number into a probability between 0 and 1.
    if z < 0:
        return math.exp(z) / (1.0 + math.exp(z))
    return 1.0 / (1.0 + math.exp(-z))

# Turn each labelled ticket into (signal_count, 0/1) training pairs.
signal_counts = []                                            # the feature x per ticket
labels = []                                                   # the answer y (1=urgent)
for text, label in training_tickets:
    signal_counts.append(count_signal_words(text))
    if label == "urgent":
        labels.append(1)
    else:
        labels.append(0)

# Train logistic regression by gradient descent (same loop shape as Lesson 14).
weight = 0.0                                                  # start the weight at 0
bias = 0.0                                                    # start the bias at 0
learning_rate = 0.3                                           # step size
iterations = 3000                                            # how many nudges
n = len(signal_counts)
for step in range(iterations):
    weight_gradient = 0.0
    bias_gradient = 0.0
    for i in range(n):
        x = signal_counts[i]                                 # this ticket's signal count
        actual = labels[i]                                   # its true 0/1 label
        predicted = sigmoid(weight * x + bias)               # predicted probability
        error = predicted - actual                           # probability minus truth
        weight_gradient += error * x                         # weight the error by x
        bias_gradient += error                               # plain error for bias
    weight -= learning_rate * (weight_gradient / n)          # step weight downhill
    bias -= learning_rate * (bias_gradient / n)              # step bias downhill

print(f"Learned model: P(urgent) = sigmoid({weight:.2f} * signal_words + {bias:.2f})")
print(f"Decision boundary at {(-bias / weight):.2f} signal words")

def urgency_probability(ticket_text):
    x = count_signal_words(ticket_text)                      # extract the feature
    return sigmoid(weight * x + bias)                        # push through the model

def predict_urgency(ticket_text):
    if urgency_probability(ticket_text) >= 0.5:              # threshold at 0.5
        return "urgent"
    return "not urgent"

Learned model: P(urgent) = sigmoid(5.90 * signal_words + -5.69)
Decision boundary at 0.97 signal words


## Trying it on brand new tickets

Neither of these two tickets appears in `training_tickets`. This checks whether the assistant generalizes to genuinely new customer messages, not just the ones it memorized.

In [4]:
incoming_tickets = [
    "My payment failed twice and I need this resolved right now",
    "Do you have this item in a larger size",
]

for ticket in incoming_tickets:
    probability = urgency_probability(ticket)                # the model's confidence
    decision = predict_urgency(ticket)                       # the yes/no call
    signals = count_signal_words(ticket)                     # why: how many signal words
    print(f"{ticket}")
    print(f"   signal words: {signals}   P(urgent)={probability:.2f}   ->  {decision}")

My payment failed twice and I need this resolved right now
   signal words: 2   P(urgent)=1.00   ->  urgent
Do you have this item in a larger size
   signal words: 0   P(urgent)=0.00   ->  not urgent


## Status of this feature

This is now a real learned classifier — **logistic regression** trained by gradient descent — not a hand-written rule. Two honest limits remain: it uses a single feature (the count of a fixed list of signal words), and it trained on only eight tickets, so it is still a small demo, not production ML.

**What comes next for this feature:** Chapter 6 (feature engineering) will let it read the *whole* ticket instead of a hand-picked word list, and Chapter 7 (evaluation metrics) will score it with precision, recall, and F1 instead of eyeballing two examples.

**Where this lives in the real project:** the same logic, as reusable functions, is in `../ticket_urgency.py`, and `../main.py` imports it. Run `python main.py` from inside `customer-support-assistant/` to see it alongside every other capability, now printing a probability for each ticket.